# Building the book recommender datasetProduces two tables, joined on **`work_id`**:| output | rows | what it is ||---|---|---|| `books.parquet` | **92,526 works** | book metadata — the **content** side || `interactions.parquet` | **775,090 ratings** | user→book ratings — the **collaborative** side |**At a glance** — full profile in §11| size | | feature coverage (after §12 backfill) | ||---|---|---|---|| works | 92,526 | description | **88.0%** || interactions | 775,090 | cover image | **97.6%** || users | 83,200 | both cover + description | 86.4% || explicit (rated 1–10) | 38.7% | genres / shelves | 96.7% / 99.6% || density | 1.0e-04 | author name | 97.5% |> **Three things to remember.** ① It is **sparse** — the median work has 2 ratings and> the median user has 1; a `users>=5 & works>=5` k-core leaves 15,899 users and 25,280> works. That is the main design constraint. ② Ratings are **skewed positive** (mean> 7.69/10) and 61% are implicit `0`s. ③ Only **808 works (0.9%)** have neither a cover> nor a description, so nearly everything is reachable by content-based methods —> weighted by ratings, just 0.7% of interactions touch a work with no cover.## Sources| file | dataset | link ||---|---|---|| `archive/Ratings.csv` | Book–Crossing (via Kaggle) | https://www.kaggle.com/datasets/arashnic/book-recommendation-dataset || `goodreads_books.json` | UCSD Book Graph — books | https://cseweb.ucsd.edu/~jmcauley/datasets/goodreads.html#datasets || `goodreads_book_authors.json` | UCSD Book Graph — authors | *(same page)* || `goodreads_book_genres_initial.json` | UCSD Book Graph — genres | *(same page)* |The UCSD Book Graph is by Mengting Wan and Julian McAuley (UCSD); cite their papers ifyou publish with it.---## Why 126,217 books?The two source datasets are unrelated releases that only meet through ISBN:- **Book–Crossing** — 1,149,780 ratings from 105,283 users over **340,200 distinct  ISBNs**. Has interactions, but almost no usable metadata (title, author, year, cover  URL — no description, no genre).- **UCSD Goodreads** — 8.6 GB, **2,360,655 books** with rich metadata, but no user  interactions in this file.A recommender doing collaborative filtering *and* content-based retrieval needs bothhalves, so the usable universe is **the intersection**: 126,217 ISBNs.**The 37% headline undersells it.** Only 37.1% of Book–Crossing's distinct ISBNs are inthe Goodreads dump, but those books carry **68.1% of all rating rows** — the matchedbooks are the popular ones, and the 63% dropped is long tail averaging 1.71 ratings perbook (median 1), largely unusable for CF anyway.| | books | share ||---|---|---|| unique ISBNs in `Ratings.csv` | 340,200 | 100% || matched on `isbn` directly | 124,065 | 36.5% || matched after ISBN-10 → 13 convert | 2,152 | 0.6% || **kept** | **126,217** | **37.1%** || dropped (not in Goodreads) | 213,983 | 62.9% |See `inspect_goodreads.ipynb` for the analysis behind these numbers.---## Why `.parquet`?Parquet is a **columnar, typed, compressed binary** table format (the pandas/Spark/DuckDB standard). CSV was the alternative and it loses badly here — measured on thisexact books table:| | CSV | Parquet ||---|---|---|| file size | 173.8 MB | **66.3 MB** || full read | 0.91 s | **0.28 s** || read 2 columns only | 0.91 s (must parse all) | **0.02 s** |Size and speed are nice, but **the real reason is types**. CSV has no schema — it storestext — so a round-trip through it:- turns `Int32` into `float64` (`ratings_count` 3 becomes `3.0`, nulls force floats);- turns our list columns into **unparseable strings**: `authors` comes back as the  literal text `"['2983296' '40075']"`, not a list.Since `authors`, `shelves`, `genres` and `similar_books` are all list-valued, CSV wouldmean re-parsing every one on load. Parquet stores them as native list types and givesthem back intact. That is also exactly the pain the raw Goodreads JSON causes — everyfield is a string — and we do not want to reintroduce it downstream.

In [ ]:
import json
import re
import time
from pathlib import Path

import pandas as pd

DATA = Path("/Users/jakubhajko/Projects/bookshelf")
BOOKS_JSON = DATA / "goodreads_books.json"          # JSON Lines, 8.6 GB, streamed
AUTHORS_JSON = DATA / "goodreads_book_authors.json"
GENRES_JSON = DATA / "goodreads_book_genres_initial.json"
RATINGS_CSV = DATA / "archive" / "Ratings.csv"
OUT_BOOKS_ED = DATA / "books_editions.parquet"        # 126,217 printings (intermediate)
OUT_INTER_ED = DATA / "interactions_editions.parquet"
OUT_BOOKS = DATA / "books.parquet"                    # 92,526 works  <- the dataset
OUT_INTER = DATA / "interactions.parquet"

# Some Book-Crossing ISBNs carry junk suffixes ("9070066882/CI"); this turns any ISBN
# into a safe filename. Used for cover files throughout (see §9).
UNSAFE = re.compile(r"[^A-Z0-9]")

## 1. Book–Crossing ratingsISBNs need normalising before they will join — the raw ones carry hyphens, spaces andlowercase `x` check digits.

In [ ]:
def norm(s):
    return str(s).strip().upper().replace("-", "").replace(" ", "")


def isbn10_to_13(i10):
    core = i10[:9]
    if not core.isdigit():
        return None
    body = "978" + core
    tot = sum(int(d) * (1 if k % 2 == 0 else 3) for k, d in enumerate(body))
    return body + str((10 - tot % 10) % 10)


ratings = pd.read_csv(RATINGS_CSV)
ratings["isbn"] = ratings["ISBN"].map(norm)
bx_unique = set(ratings["isbn"].unique())

# Goodreads sometimes carries only an isbn13, so index our ISBNs by their 13 form too.
bx_by13 = {}
for i in bx_unique:
    c = isbn10_to_13(i)
    if c:
        bx_by13.setdefault(c, []).append(i)

print(f"rating rows  : {len(ratings):,}")
print(f"unique users : {ratings['User-ID'].nunique():,}")
print(f"unique ISBNs : {len(bx_unique):,}")

## 2. One streaming pass over the 8.6 GBWe keep a record only if its ISBN is one we have ratings for, so memory holds ~126kbooks rather than 2.36M. Takes ~40 s.**Deduplication:** 202 ISBNs are hit by more than one Goodreads record (differenteditions). We keep the most useful one: *has a description* first, then highest`ratings_count`.

In [ ]:
best = {}   # bx isbn -> (rank_key, record)
t = time.perf_counter()
n = 0
with open(BOOKS_JSON, "rb") as f:
    for line in f:
        r = json.loads(line)
        keys = []
        if r["isbn"]:
            a = norm(r["isbn"])
            if a in bx_unique:
                keys.append(a)
        if r["isbn13"]:
            keys.extend(bx_by13.get(norm(r["isbn13"]), ()))
        if keys:
            desc = (r["description"] or "").strip()
            rank = (len(desc) > 0, int(r["ratings_count"] or 0))
            for k in set(keys):
                if k not in best or rank > best[k][0]:
                    best[k] = (rank, r)
        n += 1
        if n % 400_000 == 0:
            print(f"  {n:,} records  {time.perf_counter()-t:.0f}s", flush=True)

print(f"scanned {n:,} records in {time.perf_counter()-t:.0f}s")
print(f"matched books: {len(best):,}")

## 3. Author names`goodreads_books.json` stores only `author_id`s. `goodreads_book_authors.json` (829,529records) maps those to names.**22.2% of books have more than one author** (max 51), and the list mixes realco-authors with `Translator`, `Illustrator`, `Editor`, `Narrator` and so on. Goodreadsmarks the main writer with an **empty `role`**. So we keep three parallel lists plus aconvenience scalar:- `authors` — all names, original order- `author_ids` — the ids, same order- `author_roles` — `""` for the main writer, otherwise `Translator`/`Illustrator`/…- `primary_author` — first name whose role is empty (what you want for most features)

In [ ]:
author_name = {}
with open(AUTHORS_JSON, "r", encoding="utf-8") as f:
    for line in f:
        a = json.loads(line)
        author_name[a["author_id"]] = a["name"]

print(f"authors loaded: {len(author_name):,}")

## 4. Genres — and how they differ from shelvesThis is the part worth understanding, because the two look interchangeable and are not.### What a "shelf" isOn Goodreads every user files a book onto their own named **shelves** — that is thesite's tagging system. Some are workflow (`to-read`, `currently-reading`), some describethe book (`fantasy`, `historical-fiction`), some are personal (`read-in-2016`,`mom-recommended`). The `popular_shelves` field in `goodreads_books.json` is, per book,**the most-used shelf names with how many users used each**:```json"popular_shelves": [{"count": "2634", "name": "to-read"},                    {"count": "160",  "name": "fiction"}, ...]```So shelves are **crowd behaviour, not editorial metadata** — free text, unboundedvocabulary, no quality control. Across our 126k books there are **173,843 distinct shelfnames**, ~19 kept per book.### What the genre file is`goodreads_book_genres_initial.json` maps `book_id` → genre counts:```json{"book_id": "7327624", "genres": {"fantasy, paranormal": 31, "fiction": 8,                                  "mystery, thriller, crime": 1, "poetry": 1}}```Crucially, UCSD documents it as *"genre tags extracted from users' popular shelves by asimple keyword matching process."* **It is not independent information — it is acleaned-up rollup of the same shelves**, collapsed onto a fixed vocabulary of exactly**10 labels**, with the counts being how many shelvings fed each label.### Which to use| | shelves | genres ||---|---|---|| vocabulary | 173,843 free-text labels | 10 fixed labels || per book | ~19 | ~2.8 || coverage | 99.7% | 97.5% || noise | high — needs a stoplist | none || origin | raw `popular_shelves` | keyword rollup **of** shelves |**Keep both — they do different jobs, and neither is a superset of the other.**- **Genres** for anything needing a small, clean, comparable label set: filters and  facets in a UI, cold-start fallbacks, stratifying a train/test split, reporting  results by segment. Ten tidy classes.- **Shelves** for the content-based retrieval itself. The whole value of content-based  similarity is discriminating *within* a genre, and "fiction" cannot do that — 76% of  the catalogue is tagged fiction. Shelves like `cyberpunk`, `victorian`, `occult`  or `occult` are what make two books actually similar, and that signal exists only in  the long tail the 10-label rollup throws away.Together they cost a few MB. Using only genres would gut content-based retrieval; usingonly shelves would leave you with no clean labels to filter or evaluate on.

In [ ]:
wanted_ids = {r["book_id"] for _, r in best.values()}
book_genres = {}
with open(GENRES_JSON, "r", encoding="utf-8") as f:
    for line in f:
        g = json.loads(line)
        if g["book_id"] in wanted_ids and g["genres"]:
            book_genres[g["book_id"]] = sorted(
                g["genres"].items(), key=lambda kv: -kv[1])

print(f"books with >=1 genre: {len(book_genres):,} / {len(wanted_ids):,}")
print(f"distinct labels     : {len({g for v in book_genres.values() for g, _ in v})}")

## 5. The books tableEvery field is kept for a reason; everything else is dropped.| kept | why ||---|---|| `isbn` | **join key** to interactions; unique || `book_id`, `work_id` | Goodreads ids — `book_id` joins the other UCSD files, `work_id` groups editions of one work || `title`, `title_without_series` | display + text features || `description` | **the main content signal** (86% filled, median 614 chars) || `authors`, `author_ids`, `author_roles`, `primary_author` | strong content signal; roles let you ignore translators/illustrators || `genres`, `genre_counts`, `top_genre` | clean 10-label taxonomy — filters, facets, cold start, eval slices || `shelves`, `shelf_counts` | fine-grained crowd tags — the discriminative content signal || `similar_books` | Goodreads' own item-item graph, a ready-made neighbour list || `series` | series membership; strong "read next" signal || `average_rating`, `ratings_count`, `text_reviews_count` | global popularity — cold start, popularity debiasing || `bx_ratings`, `bx_explicit` | popularity **inside this dataset**, for CF min-interaction cutoffs || `num_pages`, `publication_year`, `publisher`, `language_code`, `format`, `is_ebook` | cheap features and filters || `image_url`, `url` | display in a UI |**Dropped:** `country_code`, `asin`, `kindle_asin`, `edition_information` (near-empty orretail plumbing), `publication_day`/`_month` (year suffices), `link` (byte-identical to`url` in all 2.36M records), and raw `popular_shelves` (replaced by the filtered top-20).### What we kept from `popular_shelves`, exactlyThe top **20** shelves by user count, after removing (a) a stoplist of workflow/format/ownership shelves and (b) year-like names (`read-in-2016`). Stored as two parallelarrays, descending by count: `shelves` (names) and `shelf_counts` (how many usersshelved the book under that name — use these as **weights**, not just membership).

In [ ]:
# popular_shelves is free text: real genres mixed with shelving verbs and personal
# junk. Drop the obvious non-genres; extend this set as you spot more.
NON_GENRE = {
    "to-read", "currently-reading", "default", "owned", "books-i-own", "owned-books",
    "favorites", "favourites", "favorite", "my-books", "library", "to-buy", "wishlist",
    "wish-list", "kindle", "audiobook", "audiobooks", "audio", "ebook", "ebooks",
    "e-book", "e-books", "books", "book", "series", "school", "re-read", "reread",
    "did-not-finish", "didn-t-finish", "dnf", "abandoned", "unfinished", "unread",
    "my-library", "have", "i-own", "all-books", "paperback", "hardcover", "bookshelf",
    "tbr", "own-it", "own", "home-library", "book-club", "english", "shelfari-favorites",
    "maybe", "reviewed", "audible", "wanted", "general", "stand-alone", "standalone",
}
YEARISH = re.compile(r"^(read|owned)?-?in-?\d{4}$|^\d{4}$|^read-\d{4}$")
TOP_SHELVES = 20


def genre_shelves(shelves, k=TOP_SHELVES):
    out = [(s["name"], int(s["count"])) for s in shelves
           if s["name"] not in NON_GENRE and not YEARISH.match(s["name"])]
    out.sort(key=lambda x: -x[1])
    return out[:k]


bx_counts = ratings.groupby("isbn").agg(
    bx_ratings=("Book-Rating", "size"),
    bx_explicit=("Book-Rating", lambda s: int((s > 0).sum())),
)

rows = []
for isbn, (_, r) in best.items():
    sh = genre_shelves(r["popular_shelves"])
    gen = book_genres.get(r["book_id"], [])
    ids = [a["author_id"] for a in r["authors"]]
    roles = [a["role"] for a in r["authors"]]
    names = [author_name.get(i, "") for i in ids]
    primary = next((nm for nm, ro in zip(names, roles) if ro == "" and nm), "")

    rows.append({
        "isbn": isbn,
        "book_id": r["book_id"],
        "work_id": r["work_id"],
        "title": r["title"],
        "title_without_series": r["title_without_series"],
        "description": (r["description"] or "").strip(),
        "authors": names,
        "author_ids": ids,
        "author_roles": roles,
        "primary_author": primary,
        "genres": [g for g, _ in gen],
        "genre_counts": [c for _, c in gen],
        "top_genre": gen[0][0] if gen else "",
        "shelves": [s for s, _ in sh],
        "shelf_counts": [c for _, c in sh],
        "similar_books": r["similar_books"],
        "series": r["series"],
        "average_rating": r["average_rating"],
        "ratings_count": r["ratings_count"],
        "text_reviews_count": r["text_reviews_count"],
        "num_pages": r["num_pages"],
        "publication_year": r["publication_year"],
        "publisher": r["publisher"],
        "language_code": r["language_code"],
        "format": r["format"],
        "is_ebook": r["is_ebook"],
        "isbn13": r["isbn13"],
        "image_url": r["image_url"],
        "url": r["url"],
    })

books = pd.DataFrame(rows)

# Everything arrives as a string in the raw JSON; "" means missing, not null.
books["average_rating"] = pd.to_numeric(books.average_rating, errors="coerce").astype("float32")
for c in ["ratings_count", "text_reviews_count", "num_pages", "publication_year"]:
    books[c] = pd.to_numeric(books[c], errors="coerce").astype("Int32")
books["is_ebook"] = books.is_ebook.map({"true": True, "false": False}).astype("boolean")
for c in ["publisher", "language_code", "format", "isbn13", "image_url",
          "primary_author", "top_genre"]:
    books[c] = books[c].replace("", pd.NA)

books = books.join(bx_counts, on="isbn")
books["has_description"] = books.description.str.len() > 0
books["n_shelves"] = books.shelves.map(len)
# 56% of image_urls are one shared "no cover" placeholder, not a real jacket (see §9).
books["has_cover"] = ~books.image_url.fillna("").str.contains("nophoto", case=False)

# 80 Book-Crossing "ISBNs" carry junk suffixes with characters that are illegal or
# awkward in filenames ("9070066882/CI", "0553260111>>5"). They are still valid keys,
# so we keep them as-is and derive a safe filename (UNSAFE, defined at the top).
books["cover_file"] = books.isbn.str.replace(UNSAFE, "_", regex=True) + ".jpg"
books.loc[~books.has_cover, "cover_file"] = pd.NA

print(f"books: {books.shape[0]:,} rows x {books.shape[1]} cols")
books[["isbn", "title", "primary_author", "top_genre", "bx_ratings"]].head()

## 6. The interactions tableFiltered to the kept books. **`rating == 0` means an implicit interaction** (shelved butnever scored) — 61.4% of the rows — so we keep them and flag them rather than silentlydropping two thirds of the data. Explicit-only CF filters on `is_explicit`;implicit-feedback models (ALS, BPR) can use everything.

In [ ]:
inter = (
    ratings[ratings.isbn.isin(best)]
    .rename(columns={"User-ID": "user_id", "Book-Rating": "rating"})
    [["user_id", "isbn", "rating"]]
    .reset_index(drop=True)
)
inter["is_explicit"] = inter.rating > 0
inter["user_id"] = inter.user_id.astype("int32")
inter["rating"] = inter.rating.astype("int8")

print(f"interactions : {len(inter):,} rows")
print(f"users        : {inter.user_id.nunique():,}")
print(f"books        : {inter.isbn.nunique():,}")
print(f"explicit     : {inter.is_explicit.sum():,} ({inter.is_explicit.mean():.1%})")
print(f"implicit (0) : {(~inter.is_explicit).sum():,} ({(~inter.is_explicit).mean():.1%})")
print(f"density      : {len(inter) / (inter.user_id.nunique() * inter.isbn.nunique()):.2e}")
inter.head()

## 7. Save

In [ ]:
books.to_parquet(OUT_BOOKS_ED, index=False, compression="zstd")
inter.to_parquet(OUT_INTER_ED, index=False, compression="zstd")

for p in (OUT_BOOKS_ED, OUT_INTER_ED):
    print(f"{p.name:28} {p.stat().st_size / 1e6:6.1f} MB")

## 8. Sanity checks

In [ ]:
checks = pd.DataFrame([
    ("has description",       books.has_description.sum()),
    ("has >=1 shelf",         (books.n_shelves > 0).sum()),
    ("has >=1 genre",         books.genres.map(len).gt(0).sum()),
    ("has a named author",    books.primary_author.notna().sum()),
    ("has >1 author",         books.authors.map(len).gt(1).sum()),
    ("has a real cover",      books.has_cover.sum()),
    ("has similar_books",     books.similar_books.map(len).gt(0).sum()),
    ("has publication_year",  books.publication_year.notna().sum()),
    ("in a series",           books.series.map(len).gt(0).sum()),
], columns=["feature", "books"])
checks["coverage"] = (checks.books / len(books)).map("{:.1%}".format)
checks["books"] = checks.books.map("{:,}".format)

assert inter.isbn.isin(books.isbn).all(), "orphan interaction"
assert books.isbn.is_unique, "duplicate isbn"
print("join integrity OK — every interaction has a book, every isbn unique\n")
checks

In [ ]:
print("genre distribution (books may carry several):")
g = pd.Series([g for gs in books.genres for g in gs]).value_counts()
print((g.to_frame("books").assign(
    share=lambda d: (d.books / len(books)).map("{:.1%}".format))).to_string())

In [ ]:
# What a finished row looks like.
r = books[books.authors.map(len) > 1].iloc[0]
print(f"{r.title}\n  primary : {r.primary_author}")
print(f"  authors : {list(zip(list(r.authors), list(r.author_roles)))}")
print(f"  genres  : {list(zip(list(r.genres), list(r.genre_counts)))}")
print(f"  shelves : {list(zip(list(r.shelves), list(r.shelf_counts)))[:8]}")

## 9. Cover images`image_url` is on every row, but **56.1% of them are not real covers**. They are onesingle shared placeholder:```https://s.gr-assets.com/assets/nophoto/book/111x148-bcc042a9c91a29c1d680899eff700a03.png```That exact URL appears 70,823 times. Downloading blindly would fetch the same grey"no cover" PNG seventy thousand times and leave you unable to tell a real jacket from amissing one. So we filter it out — the `has_cover` column above — and fetch only the**55,394 books with a genuine cover** (54,693 unique URLs; 701 are shared by two ISBNsof the same book, so we fetch once and copy).| | ||---|---|| files to fetch | 54,693 unique URLs → 55,394 files || average size | ~11 KB (measured) || total on disk | **~0.6 GB** || runtime | ~45–60 min at 8 workers |**Pick your resolution first.** The URLs carry a size code (`.../books/1328768789m/…`)and swapping it re-sizes the fetch. Measured on one cover:| `COVER_SIZE` | pixels | per file | 55,394 files ||---|---|---|---|| `"s"` | 47×75 | 2.3 KB | ~0.13 GB || `"m"` *(default)* | 98×155 | 9 KB | **~0.6 GB** || `"l"` | 300×475 | 62 KB | ~3.5 GB |The stored `image_url` is the `m` variant. `m` is fine for a UI thumbnail, but 98 px ismarginal if you plan to run a CNN over the jackets for visual similarity — choose `"l"`now rather than re-downloading later.The downloader is **resumable** — it skips files already on disk, so an interrupted runcosts nothing. It writes to a `.part` file and renames only on success, so you neverresume onto a truncated image. It also verifies the JPEG magic bytes rather thantrusting the response, so an error page never lands on disk as a `.jpg`.**Naming: use the `cover_file` column, not `f"{isbn}.jpg"`.** For 99.94% of rows theyare the same thing, but **80 Book–Crossing "ISBNs" carry junk suffixes** —`9070066882/CI`, `0553260111>>5`, `0684835355(PB`. They are genuine matches (theISBN-13 conversion reads only the first 9 digits, so the junk is ignored) and genuinekeys, so we keep them intact and derive the filename by replacing anything outside`[A-Z0-9]` with `_`. Six of them contain a **`/`**, which silently means "subdirectory"— the first run of this notebook crashed on exactly that.A `covers_manifest.parquet` records `isbn`, `cover_file`, `image_url` and `status` perbook, so failures are inspectable and the isbn→file mapping is explicit.> `SAMPLE_N` below is set to 24 so a first run finishes in seconds. **Set it to `None`> to fetch all 55,394.** It is a polite 8 workers with retry/backoff; do not raise it> much — this is someone else's CDN.

In [ ]:
import shutil
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor

COVERS = DATA / "covers"
MANIFEST = DATA / "covers_manifest.parquet"
COVERS.mkdir(exist_ok=True)

WORKERS = 8
TIMEOUT = 20
RETRIES = 3
SAMPLE_N = 24        # <-- set to None to download everything
COVER_SIZE = "m"     # "s" 47x75 | "m" 98x155 (~0.6 GB) | "l" 300x475 (~3.5 GB)
UA = "bookshelf-dataset-builder/1.0 (personal research project)"

# The size code sits in the path: .../books/1328768789m/287149.jpg
SIZE_RE = re.compile(r"/(\d+)[sml]/")

targets = books.loc[books.has_cover,
                    ["isbn", "cover_file", "image_url"]].reset_index(drop=True)
targets["image_url"] = targets.image_url.str.replace(
    SIZE_RE, rf"/\g<1>{COVER_SIZE}/", regex=True)
if SAMPLE_N:
    targets = targets.head(SAMPLE_N)

# 701 ISBNs share a URL with another edition: fetch each URL once, copy for the rest.
by_url = targets.groupby("image_url").apply(
    lambda d: list(zip(d.isbn, d.cover_file)), include_groups=False)
print(f"books with a real cover : {books.has_cover.sum():,}")
print(f"this run: {len(targets):,} files from {len(by_url):,} unique URLs"
      f"{'  (SAMPLE)' if SAMPLE_N else ''}")

In [ ]:
def fetch(url, pairs):
    "Download one URL, write it for every isbn that shares it."
    dests = [COVERS / fn for _, fn in pairs]
    todo = [d for d in dests if not (d.exists() and d.stat().st_size > 0)]
    if not todo:
        return ("cached", len(dests), 0)

    delay = 1.0
    for attempt in range(RETRIES):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": UA})
            with urllib.request.urlopen(req, timeout=TIMEOUT) as resp:
                data = resp.read()
        except urllib.error.HTTPError as e:
            if e.code in (403, 404, 410):     # permanent: do not retry
                return (f"http-{e.code}", 0, 0)
            if attempt == RETRIES - 1:
                return (f"http-{e.code}", 0, 0)
        except Exception as e:
            if attempt == RETRIES - 1:
                return (type(e).__name__, 0, 0)
        else:
            if not data.startswith(b"\xff\xd8"):   # not a JPEG -> error page
                return ("not-jpeg", 0, 0)
            first = todo[0]
            tmp = first.with_suffix(".part")       # atomic: write then rename
            tmp.write_bytes(data)
            tmp.rename(first)
            for d in todo[1:]:
                shutil.copyfile(first, d)
            return ("ok", len(todo), len(data))
        time.sleep(delay)
        delay *= 2
    return ("failed", 0, 0)


t = time.perf_counter()
results, done, nbytes = [], 0, 0
with ThreadPoolExecutor(max_workers=WORKERS) as pool:
    futs = {pool.submit(fetch, u, p): (u, p) for u, p in by_url.items()}
    for k, f in enumerate(futs, 1):
        try:
            status, nfiles, size = f.result()
        except Exception as e:                  # never let one URL kill the run
            status, nfiles, size = f"crash-{type(e).__name__}", 0, 0
        url, pairs = futs[f]
        nbytes += size
        done += nfiles
        for i, fn in pairs:
            results.append({"isbn": i, "cover_file": fn,
                            "image_url": url, "status": status})
        if k % 500 == 0:
            el = time.perf_counter() - t
            print(f"  {k:,}/{len(futs):,} urls  {done:,} files  "
                  f"{nbytes/1e6:.0f} MB  {el:.0f}s  eta {el/k*(len(futs)-k)/60:.0f}m",
                  flush=True)

print(f"\n{done:,} files written, {nbytes/1e6:.1f} MB, "
      f"{time.perf_counter()-t:.0f}s")

In [ ]:
manifest = pd.DataFrame(results)
manifest["path"] = "covers/" + manifest.cover_file

# Merge with any previous run so the manifest reflects everything on disk.
if MANIFEST.exists():
    old = pd.read_parquet(MANIFEST)
    manifest = (pd.concat([old, manifest])
                  .drop_duplicates("isbn", keep="last").reset_index(drop=True))
manifest.to_parquet(MANIFEST, index=False)

print(manifest.status.value_counts().to_string())
on_disk = list(COVERS.glob("*.jpg"))
print(f"\nfiles on disk : {len(on_disk):,}")
print(f"bytes on disk : {sum(p.stat().st_size for p in on_disk)/1e6:.1f} MB")
print(f"stray .part   : {len(list(COVERS.glob('*.part')))}")

In [ ]:
# Pairing check: every downloaded file maps back to a book row.
ok = manifest[manifest.status.isin(["ok", "cached"])]
assert ok.isbn.isin(books.isbn).all(), "cover with no matching book"
print(f"{len(ok):,} covers pair cleanly with book rows\n")

sample = books[books.isbn.isin(ok.isbn)].head(3)
for _, r in sample.iterrows():
    p = COVERS / r.cover_file
    print(f"{r.isbn}  {p.stat().st_size/1024:5.1f} KB  {r.title[:44]}")

### Using the covers```pythonfrom pathlib import PathCOVERS = Path("covers")def cover_path(row):    "row: a books.parquet row. Returns a Path, or None if there is no cover."    if not row.has_cover:        return None    p = COVERS / row.cover_file          # NOT f"{row.isbn}.jpg" — see above    return p if p.exists() else None     # None here = not downloaded yet````books.has_cover` tells you whether a cover *exists upstream*; the file being presenttells you whether it has been **fetched**. They differ until a full run completes, whichis why the manifest is worth keeping. To find what is still missing:```pythonmissing = books.loc[books.has_cover & ~books.isbn.isin(    pd.read_parquet("covers_manifest.parquet").query("status in ['ok','cached']").isbn)]```Re-running the download cell with `SAMPLE_N = None` picks up exactly those and skipseverything already on disk.

## 10. Merging editions into worksEverything above is keyed by **ISBN — a printing**, not a book. 33,691 rows (26.7%) areextra editions of a work already in the table: *Frankenstein* appears 55 times, *TheLittle Prince* 50, *Pride and Prejudice* 47.That is a problem, not a curiosity:- **It splits the rating signal.** Frankenstein's 409 ratings are spread over 55 ISBNs  (66, 37, 31, 30, 28 …), so collaborative filtering sees 55 obscure items instead of  one popular book. **66.4% of all ratings** sit on multi-edition works.- **It produces duplicate recommendations.** "Top 10 for you" becomes Frankenstein  (Penguin), Frankenstein (Dover), Frankenstein (Bantam) …- **It poisons content-based retrieval.** 55 near-identical descriptions mean a work's  nearest neighbours are mostly itself.So we collapse on `work_id` → **92,526 works**. Effect on the signal:| | by ISBN | by work ||---|---|---|| items | 126,217 | 92,526 || mean ratings/item | 6.20 | **8.46 (+36%)** || items with ≥10 ratings | 12.9% | **15.5%** || density | 7.5e-05 | **1.0e-04** |### Which edition represents the workEditions are ranked and the top one supplies identity (title, ISBN, cover, publisher).The order is: **English first** → has a cover → has a description → most ratings in*this* dataset → most ratings on Goodreads. English wins because 16.4% of multi-editionworks mix languages (an English and a German printing of the same book), and a Germandescription in an English catalogue is worse than useless for text features.### But fields are merged, not just copiedTaking one edition wholesale would throw away data, so the good fields are pooled:| field | rule ||---|---|| `description` | **longest** across editions, ignoring ones in a *different* known language || `cover_file` / `isbn` | representative's — or the best edition that *has* a cover, if it doesn't || `shelves`, `genres` | **counts summed across all editions**, re-sorted — strictly more signal than any one edition || `similar_books`, `series` | union across editions || `ratings_count`, `text_reviews_count` | summed || `average_rating` | mean weighted by each edition's `ratings_count` || `publication_year` | **earliest** of the merged printings — see caveat below || `n_editions`, `edition_isbns` | kept, so nothing is lost |### No interactions are lostEvery rating is carried over. The only rows that disappear are **genuine duplicates** —the same user rating the same book through two different editions (7,182 pairs, 7,837rows, 1.0%). Those collapse with **`max`**, chosen from the data: 2,159 of those pairsmix a real score with an implicit `0`, where `max` correctly keeps the score, and amongpairs with two real scores 52.7% are identical with a mean spread under 1 point.`books_editions.parquet` keeps the full 126,217-row edition table either way.> **`publication_year` is not the original publication date.** Goodreads stores the> year of each *printing*, so the earliest we can see is the oldest reprint in this> dataset — Frankenstein comes out as **1965**, not 1818. Treat it as "oldest edition we> hold", useful for recency filters, useless as a work's true age. The UCSD> `goodreads_book_works.json` file carries `original_publication_year` if you need the> real thing.

In [ ]:
# --- canonical language, so "eng"/"en"/"en-US" are one thing ---
LANG = {"eng": "en", "en": "en", "ger": "de", "deu": "de", "de": "de", "fre": "fr",
        "fra": "fr", "fr": "fr", "spa": "es", "es": "es", "ita": "it", "it": "it",
        "por": "pt", "pt": "pt", "dut": "nl", "nld": "nl", "nl": "nl", "jpn": "ja",
        "ja": "ja", "swe": "sv", "sv": "sv", "pol": "pl", "pl": "pl", "rus": "ru",
        "ru": "ru"}


def canon_lang(x):
    if pd.isna(x):
        return None
    s = str(x).split("-")[0].lower()
    return LANG.get(s, s)


ed = books.copy()
ed["lang"] = ed.language_code.map(canon_lang)

# --- rank editions within each work; row 1 of each group is the representative ---
ed["_en"] = (ed.lang == "en").astype(int)
ed["_cov"] = ed.has_cover.astype(int)
ed["_desc"] = ed.has_description.astype(int)
ed["_bx"] = ed.bx_ratings.fillna(0).astype(int)
ed["_rc"] = ed.ratings_count.fillna(0).astype(int)

RANK = ["_en", "_cov", "_desc", "_bx", "_rc"]
ed = ed.sort_values(["work_id"] + RANK + ["isbn"],
                    ascending=[True] + [False] * len(RANK) + [True])

# drop_duplicates takes whole rows; groupby().first() would mix fields across editions.
rep = ed.drop_duplicates("work_id", keep="first").set_index("work_id")
sizes = ed.groupby("work_id").size()
print(f"editions {len(ed):,} -> works {len(rep):,}")
print(f"multi-edition works: {(sizes > 1).sum():,}")

In [ ]:
from collections import Counter


def merge_lists(series):
    "Union of list-valued cells, order preserved, deduplicated."
    out, seen = [], set()
    for v in series:
        for x in (v if v is not None else []):
            if x not in seen:
                seen.add(x)
                out.append(x)
    return out


def top_counted(names_col, counts_col, k=None):
    "Sum counts for the same label across editions, return (labels, counts) desc."
    c = Counter()
    for names, counts in zip(names_col, counts_col):
        for nm, ct in zip(list(names or []), list(counts or [])):
            c[nm] += int(ct)
    items = c.most_common(k)
    return [n for n, _ in items], [int(v) for _, v in items]


PLAUSIBLE_YEAR = 1400
works = rep.copy()
multi = sizes[sizes > 1].index
t = time.perf_counter()

for wid, g in ed[ed.work_id.isin(multi)].groupby("work_id", sort=False):
    r = rep.loc[wid]

    # description: longest, excluding editions in a different *known* language
    cand = g[g.has_description]
    if len(cand):
        same = cand[cand.lang.isna() | (cand.lang == r.lang)] if pd.notna(r.lang) else cand
        pool = same if len(same) else cand
        works.at[wid, "description"] = pool.loc[pool.description.str.len().idxmax(),
                                                "description"]

    # cover: representative's if it has one, else the best-ranked edition that does
    if not r.has_cover:
        cov = g[g.has_cover]
        if len(cov):
            best = cov.iloc[0]
            works.at[wid, "cover_file"] = best.cover_file
            works.at[wid, "image_url"] = best.image_url
            works.at[wid, "isbn"] = best.isbn      # keep isbn <-> cover consistent

    # pooled tag signal
    works.at[wid, "shelves"], works.at[wid, "shelf_counts"] = top_counted(
        g.shelves, g.shelf_counts, TOP_SHELVES)
    works.at[wid, "genres"], works.at[wid, "genre_counts"] = top_counted(
        g.genres, g.genre_counts)
    works.at[wid, "similar_books"] = merge_lists(g.similar_books)
    works.at[wid, "series"] = merge_lists(g.series)

    # popularity summed; average_rating weighted by each edition's ratings_count
    rc = g.ratings_count.fillna(0).astype("int64")
    works.at[wid, "ratings_count"] = int(rc.sum())
    works.at[wid, "text_reviews_count"] = int(g.text_reviews_count.fillna(0).sum())
    if rc.sum() > 0:
        works.at[wid, "average_rating"] = float(
            (g.average_rating.fillna(0) * rc).sum() / rc.sum())

    yrs = g.publication_year.dropna()
    yrs = yrs[yrs >= PLAUSIBLE_YEAR]
    if len(yrs):
        works.at[wid, "publication_year"] = int(yrs.min())

print(f"merged {len(multi):,} multi-edition works in {time.perf_counter()-t:.0f}s")

In [ ]:
# provenance: how many printings fed each work, and which
works["n_editions"] = sizes
works["edition_isbns"] = ed.groupby("work_id").isbn.apply(list)
works["top_genre"] = works.genres.map(lambda g: g[0] if len(g) else pd.NA)
works["has_description"] = works.description.fillna("").str.len() > 0
works["has_cover"] = works.cover_file.notna()
works["n_shelves"] = works.shelves.map(len)

works = works.drop(columns=RANK + ["lang", "bx_ratings", "bx_explicit"]).reset_index()
print(f"works: {works.shape[0]:,} rows x {works.shape[1]} cols")

### Interactions, remapped to works

In [ ]:
iw = inter.merge(books[["isbn", "work_id"]], on="isbn", how="left")
assert iw.work_id.notna().all(), "interaction with no work"

before = len(iw)
inter_w = (iw.groupby(["user_id", "work_id"], as_index=False, sort=False)
             .rating.max())
inter_w["is_explicit"] = inter_w.rating > 0
inter_w["rating"] = inter_w.rating.astype("int8")

# in-dataset popularity, recomputed at work level
wc = inter_w.groupby("work_id").agg(
    bx_ratings=("rating", "size"),
    bx_explicit=("is_explicit", "sum"))
works = works.join(wc, on="work_id")
works["bx_ratings"] = works.bx_ratings.fillna(0).astype("int32")
works["bx_explicit"] = works.bx_explicit.fillna(0).astype("int32")

print(f"interactions {before:,} -> {len(inter_w):,}  "
      f"(-{before-len(inter_w):,} duplicate user-work pairs, "
      f"{(before-len(inter_w))/before:.2%})")
print(f"users  : {inter_w.user_id.nunique():,}   (was {inter.user_id.nunique():,})")
print(f"works  : {inter_w.work_id.nunique():,}")
print(f"explicit ratings kept: {inter_w.is_explicit.sum():,} "
      f"(was {inter.is_explicit.sum():,})")

In [ ]:
works.to_parquet(OUT_BOOKS, index=False, compression="zstd")
inter_w.to_parquet(OUT_INTER, index=False, compression="zstd")

# Nothing was lost: every user survives, every rated work is present.
assert set(inter_w.user_id) == set(inter.user_id), "lost a user"
assert inter_w.work_id.isin(works.work_id).all(), "orphan interaction"
assert works.work_id.is_unique, "duplicate work"
assert works.n_editions.sum() == len(books), "edition count mismatch"
print("integrity OK — no user lost, no orphan interaction, editions accounted for\n")

for p in (OUT_BOOKS, OUT_INTER):
    print(f"{p.name:28} {p.stat().st_size/1e6:6.1f} MB")

cmp = pd.DataFrame({
    "editions": [len(books), books.has_description.mean(), books.has_cover.mean(),
                 len(inter)],
    "works": [len(works), works.has_description.mean(), works.has_cover.mean(),
              len(inter_w)],
}, index=["rows", "has description", "has cover", "interactions"])
cmp.loc[["has description", "has cover"]] = cmp.loc[
    ["has description", "has cover"]].map(lambda v: f"{v:.1%}")
cmp

In [ ]:
# Frankenstein: 55 printings -> 1 work
f = works[works.title.str.startswith("Frankenstein", na=False)].nlargest(1, "n_editions")
r = f.iloc[0]
print(f"{r.title}  ({r.primary_author})")
print(f"  editions merged : {r.n_editions}")
print(f"  ratings         : {r.bx_ratings} (was split across {r.n_editions} items)")
print(f"  goodreads count : {r.ratings_count:,}   avg {r.average_rating:.2f}")
print(f"  first published : {r.publication_year}")
print(f"  description     : {len(r.description):,} chars")
print(f"  genres          : {list(zip(list(r.genres), list(r.genre_counts)))[:4]}")
print(f"  shelves         : {list(zip(list(r.shelves), list(r.shelf_counts)))[:5]}")
print(f"  cover           : {r.cover_file}")

## 11. Dataset profile — what you actually haveEverything below is computed from the saved files, so re-running keeps it honest.

In [ ]:
W = pd.read_parquet(OUT_BOOKS)
I = pd.read_parquet(OUT_INTER)
N = len(W)

cov = pd.DataFrame(
    [(lbl, m.sum(), m.mean()) for lbl, m in [
        ("description",       W.has_description),
        ("cover image",       W.has_cover),
        ("BOTH cover + desc", W.has_description & W.has_cover),
        ("NEITHER",           ~W.has_description & ~W.has_cover),
        (">=1 genre",         W.genres.map(len) > 0),
        (">=1 shelf",         W.n_shelves > 0),
        ("named author",      W.primary_author.notna()),
        ("similar_books",     W.similar_books.map(len) > 0),
        ("in a series",       W.series.map(len) > 0),
        ("publication_year",  W.publication_year.notna()),
        ("num_pages",         W.num_pages.notna()),
        ("language known",    W.language_code.notna()),
    ]], columns=["feature", "works", "coverage"])
cov["works"] = cov.works.map("{:,}".format)
cov["coverage"] = cov.coverage.map("{:.1%}".format)
cov

**Content features are strong across the board** (these figures are post-backfill —§12 raised covers from 46.7% to 97.6% and descriptions from 84.6% to 88.0%). Textretrieval is well supported at 88% descriptions and ~97% genres/shelves/author, andcovers are now near-complete. Only **808 works (0.9%) have neither** cover nordescription — those are reachable by collaborative signal alone.

In [ ]:
pw = W.bx_ratings
pu = I.groupby("user_id").size()

dist = pd.DataFrame({
    "ratings per WORK": [f"{pw.mean():.2f}", f"{pw.median():.0f}", f"{pw.quantile(.9):.0f}",
                         f"{pw.quantile(.99):.0f}", f"{pw.max():,}"],
    "ratings per USER": [f"{pu.mean():.2f}", f"{pu.median():.0f}", f"{pu.quantile(.9):.0f}",
                         f"{pu.quantile(.99):.0f}", f"{pu.max():,}"],
}, index=["mean", "median", "p90", "p99", "max"])
print(dist.to_string(), "\n")

thr = pd.DataFrame([(k, (pw >= k).sum(), (pw >= k).mean(), (pu >= k).sum(), (pu >= k).mean())
                    for k in (1, 2, 5, 10, 20, 50)],
                   columns=["k", "works >=k", "share", "users >=k", "share "])
thr["share"] = thr["share"].map("{:.1%}".format)
thr["share "] = thr["share "].map("{:.1%}".format)
thr["works >=k"] = thr["works >=k"].map("{:,}".format)
thr["users >=k"] = thr["users >=k"].map("{:,}".format)
print(thr.to_string(index=False))
print(f"\nevery work has >=1 rating: {(pw == 0).sum() == 0}")
print(f"top 1% of works hold {pw.nlargest(int(N*.01)).sum()/pw.sum():.1%} of all interactions")
print(f"density: {len(I)/(len(pu)*N):.2e}")

**This is a long-tail dataset and that is the main design constraint.** Median work: 2ratings. Median user: **1** rating. Only 21% of users have 5+ interactions, and the top1% of works absorb 28% of all activity. Pure collaborative filtering will do nothing formost users — the content side is what carries cold start.

In [ ]:
ex = I[I.is_explicit]
print(f"explicit (rated 1-10): {I.is_explicit.sum():,} ({I.is_explicit.mean():.1%})")
print(f"implicit (rating 0)  : {(~I.is_explicit).sum():,} ({(~I.is_explicit).mean():.1%})")
print(f"explicit mean {ex.rating.mean():.2f}  median {ex.rating.median():.0f}\n")
vc = ex.rating.value_counts().sort_index()
for r, c in vc.items():
    print(f"  {r:>2} {'#' * int(60 * c / vc.max()):<60} {c:>6,}")

**Ratings are skewed high and lumpy.** Mean 7.69, median 8; scores of 1–4 are barely 4%of explicit ratings. Note the spike at **5** (more common than 6) — people round to themid-point. Do not assume a symmetric scale: centre per user, or treat rating as ordinal.

In [ ]:
def kcore(df, ku, kw):
    d = df
    for _ in range(20):
        n0 = len(d)
        d = d[d.groupby("user_id").user_id.transform("size") >= ku]
        d = d[d.groupby("work_id").work_id.transform("size") >= kw]
        if len(d) == n0:
            break
    return d


rows = [("raw (all)", I), ("users>=5 & works>=5", kcore(I, 5, 5)),
        ("users>=10 & works>=10", kcore(I, 10, 10)),
        ("explicit only", ex), ("explicit, >=5 & >=5", kcore(ex, 5, 5))]
core = pd.DataFrame([(lbl, len(d), d.user_id.nunique(), d.work_id.nunique(),
                      len(d) / (d.user_id.nunique() * d.work_id.nunique()))
                     for lbl, d in rows],
                    columns=["subset", "rows", "users", "works", "density"])
for c in ["rows", "users", "works"]:
    core[c] = core[c].map("{:,}".format)
core["density"] = core["density"].map("{:.1e}".format)
core

**Plan your training set from this table.** The full matrix is too sparse to train ondirectly; a `users>=5 & works>=5` k-core keeps 72% of the interactions while shrinkingto 15,899 users × 25,280 works and lifting density 14×. Explicit-only is far smaller —143k ratings over 8,213 users — so if you want explicit feedback, expect a much smallerproblem than the headline 775k suggests.

## 12. Backfilling the gapsAfter merging, 14,207 works have no description and 49,306 have no cover. Two sourcesfill most of that, in order of cost.**Stage A — free, from the 8.6 GB already on disk.** The merge pooled fields across theeditions *in our 126k*, but the dump holds 2.36M books, including printings of our worksthat no Book–Crossing user ever rated. Those are still untapped: one local pass finds adescription for 3,065 works and a cover URL for 12,573.**Stage B — Open Library for the residual covers.** Measured on a random sample of theactual residual: **92% of missing covers** come back, versus **24% for descriptions**.Google Books is not usable — still HTTP 429 unauthenticated, and its ~1,000/day keyedquota would need ~11 days.**We do Stage A for both fields, and Stage B for covers only.** Descriptions aredeliberately left at Stage A: the works still missing one hold just **5.4% of allinteractions** (median 1 rating each), so ~2 hours of requests would buy ~2.8pp ofcoverage on books almost nobody has read. Covers are the opposite — works with no covercarry **31.2% of all interactions**, so it is the highest-value gap in the dataset.**Nothing is overwritten.** Existing descriptions and covers are never touched, onlyempty ones filled; `covers/` only gains files; `books_editions.parquet` is untouched.Two provenance columns record where every value came from:| `description_source` / `cover_source` | meaning ||---|---|| `goodreads` | from the book's own edition (original) || `goodreads-other-edition` | Stage A — a sibling printing in the dump || `openlibrary` | Stage B || *(null)* | still missing |> Open Library covers are user-contributed and vary in quality, unlike the uniform> Goodreads thumbnails. That is exactly why the source is recorded per row rather than> merged in silently — you can always exclude them.### What it actually producedRun on 2026-08-04. Stage A took 38 s to scan plus 15 min to fetch; Stage B took 91 minat 8 workers with **no throttling and no failures** — Open Library returned a usablecover for **34,468 of 36,733 (93.8%)**, against the 92% the 25-item probe predicted.| source | descriptions | | covers | ||---|---|---|---|---|| `goodreads` (original) | 78,319 | 84.6% | 43,220 | 46.7% || `goodreads-other-edition` | +3,065 | +3.3% | +12,573 | +13.6% || `openlibrary` | — | — | **+34,468** | **+37.3%** || **total** | **81,384** | **88.0%** | **90,261** | **97.6%** || still missing | 11,142 | 12.0% | **2,265** | **2.4%** |**Covers went 46.7% → 97.6%**, and works with *neither* cover nor description fell from10.0% to **0.9% (808 works)**. Weighted by ratings the change is starker: interactionstouching a work with no cover dropped from **31.2% to 0.7%**.`covers/` now holds **102,435 files, 0.98 GB** (55,394 edition covers from §9 plus47,041 added here). Every original value was preserved — the `goodreads` counts aboveare unchanged from before the backfill.

In [ ]:
BACKFILL = True          # set False to skip this section entirely
OL_COVERS = True         # Stage B (~1.4 h at 8 workers); Stage A alone is ~10 min
OL_URL = "https://covers.openlibrary.org/b/isbn/{}-M.jpg?default=false"

works = pd.read_parquet(OUT_BOOKS)
works["description_source"] = works.has_description.map({True: "goodreads"}).astype("object")
works["cover_source"] = works.has_cover.map({True: "goodreads"}).astype("object")

gap_d = works.loc[~works.has_description, ["work_id", "isbn"]]
gap_c = works.loc[~works.has_cover, ["work_id", "isbn"]]
print(f"missing description: {len(gap_d):,}")
print(f"missing cover      : {len(gap_c):,}")

### Stage A — sibling editions in the dump

In [ ]:
if BACKFILL:
    want_d, want_c = set(gap_d.work_id), set(gap_c.work_id)
    sib_desc, sib_cover = {}, {}
    t = time.perf_counter()
    with open(BOOKS_JSON, "rb") as f:
        for line in f:
            r = json.loads(line)
            k = r["work_id"]
            if k in want_d:
                d = (r["description"] or "").strip()
                if len(d) > len(sib_desc.get(k, "")):
                    sib_desc[k] = d
            if k in want_c:
                u = r["image_url"] or ""
                if "nophoto" not in u and k not in sib_cover:
                    sib_cover[k] = SIZE_RE.sub(rf"/\g<1>{COVER_SIZE}/", u)
    sib_desc = {k: v for k, v in sib_desc.items() if v}
    print(f"scanned in {time.perf_counter()-t:.0f}s")
    print(f"sibling descriptions found: {len(sib_desc):,}")
    print(f"sibling cover URLs found  : {len(sib_cover):,}")

In [ ]:
if BACKFILL:
    # descriptions: fill in place, never overwrite
    idx = works.set_index("work_id")
    for wid, d in sib_desc.items():
        idx.at[wid, "description"] = d
        idx.at[wid, "description_source"] = "goodreads-other-edition"
    works = idx.reset_index()

    # covers: download the sibling image, named by the work's representative ISBN
    todo = gap_c[gap_c.work_id.isin(sib_cover)].copy()
    todo["cover_file"] = todo.isbn.str.replace(UNSAFE, "_", regex=True) + ".jpg"
    todo["url"] = todo.work_id.map(sib_cover)
    assert not todo.cover_file.isin(
        set(p.name for p in COVERS.glob("*.jpg"))).any(), "would overwrite an existing cover"

    by_url = todo.groupby("url").apply(
        lambda d: list(zip(d.isbn, d.cover_file)), include_groups=False)
    t = time.perf_counter()
    got = {}
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futs = {pool.submit(fetch, u, p): p for u, p in by_url.items()}
        for k, f in enumerate(futs, 1):
            try:
                status, nfiles, _ = f.result()
            except Exception:
                status, nfiles = "crash", 0
            if status in ("ok", "cached"):
                for isbn, fn in futs[f]:
                    got[isbn] = fn
            if k % 2000 == 0:
                print(f"  {k:,}/{len(futs):,}  {time.perf_counter()-t:.0f}s", flush=True)
    print(f"sibling covers downloaded: {len(got):,} in {time.perf_counter()-t:.0f}s")

    m = works.isbn.isin(got)
    works.loc[m, "cover_file"] = works.loc[m, "isbn"].map(got)
    works.loc[m, "cover_source"] = "goodreads-other-edition"
    works["has_cover"] = works.cover_file.notna()
    works["has_description"] = works.description.fillna("").str.len() > 0
    print(f"coverage now — description {works.has_description.mean():.1%}, "
          f"cover {works.has_cover.mean():.1%}")

### Stage B — Open Library covers

In [ ]:
if BACKFILL and OL_COVERS:
    resid = works.loc[~works.has_cover, ["isbn"]].copy()
    resid["cover_file"] = resid.isbn.str.replace(UNSAFE, "_", regex=True) + ".jpg"
    print(f"residual covers to try: {len(resid):,}  (~92% expected, ~1.4 h)")

    t = time.perf_counter()
    got_ol = {}
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        futs = {pool.submit(fetch, OL_URL.format(i), [(i, fn)]): (i, fn)
                for i, fn in zip(resid.isbn, resid.cover_file)}
        for k, f in enumerate(futs, 1):
            try:
                status, nfiles, _ = f.result()
            except Exception:
                status = "crash"
            if status in ("ok", "cached"):
                i, fn = futs[f]
                got_ol[i] = fn
            if k % 2000 == 0:
                el = time.perf_counter() - t
                print(f"  {k:,}/{len(futs):,}  {len(got_ol):,} found  {el/60:.0f}m  "
                      f"eta {el/k*(len(futs)-k)/60:.0f}m", flush=True)
    print(f"\nOpen Library covers: {len(got_ol):,} / {len(resid):,} "
          f"({len(got_ol)/max(len(resid),1):.1%}) in {(time.perf_counter()-t)/60:.0f}m")

    m = works.isbn.isin(got_ol)
    works.loc[m, "cover_file"] = works.loc[m, "isbn"].map(got_ol)
    works.loc[m, "cover_source"] = "openlibrary"
    works["has_cover"] = works.cover_file.notna()

In [ ]:
if BACKFILL:
    works.to_parquet(OUT_BOOKS, index=False, compression="zstd")

    disk = {p.name for p in COVERS.glob("*.jpg")}
    assert works.loc[works.has_cover, "cover_file"].isin(disk).all(), "cover file missing"
    assert works.work_id.is_unique
    print("integrity OK — every has_cover row points at a file on disk\n")

    prov = pd.DataFrame({
        "description": works.description_source.fillna("(missing)").value_counts(),
        "cover": works.cover_source.fillna("(missing)").value_counts(),
    }).fillna(0).astype(int)
    prov["desc %"] = (prov.description / len(works)).map("{:.1%}".format)
    prov["cover %"] = (prov.cover / len(works)).map("{:.1%}".format)
    print(prov.to_string())
    print(f"\ndescription coverage: {works.has_description.mean():.1%}")
    print(f"cover coverage      : {works.has_cover.mean():.1%}")

## 13. Loading it later```pythonimport pandas as pdbooks = pd.read_parquet("books.parquet")          # 92,526 worksinter = pd.read_parquet("interactions.parquet")   # 775,090 ratingsdf    = inter.merge(books, on="work_id")```| file | rows | key | what ||---|---|---|---|| `books.parquet` | 92,526 | `work_id` | **the dataset** — one row per actual book || `interactions.parquet` | 775,090 | `user_id`+`work_id` | **the dataset** || `books_editions.parquet` | 126,217 | `isbn` | every printing, pre-merge || `interactions_editions.parquet` | 782,927 | `user_id`+`isbn` | pre-merge |Notes for whoever picks this up:- **`work_id` is the primary key.** `isbn` is still there but it is now the  *representative* printing — the one whose cover and title the work displays. Use it to  find the cover; do not treat it as unique-per-book across the two edition files.- **`n_editions` and `edition_isbns`** record what was merged, so any collapse can be  traced back or undone against `books_editions.parquet`.- **`bx_ratings` / `bx_explicit` are work-level counts**, recomputed after merging — they  are the numbers to use for min-interaction cutoffs.- The list columns (`authors`, `genres`, `shelves`, `similar_books`, …) come back from  parquet as **numpy arrays**, not Python lists. `len()`, indexing and iteration work as  expected; `+` concatenates elementwise instead of appending.- **`similar_books` holds Goodreads `book_id`s, not ISBNs.** To keep only neighbours  present here: `id2isbn = books.drop_duplicates("book_id").set_index("book_id").isbn`.- `shelf_counts` and `genre_counts` are **user counts, not scores** — they scale with a  book's popularity, so normalise per book (e.g. divide by the row's max) before using  them as features.- The dropped 63% of Book–Crossing ISBNs are long tail (median 1 rating each). If you  need them for a popularity baseline they remain in `archive/Ratings.csv`.